# Push-T: train, evaluate and visualize one policy

*Author: Mohammad Al-Jarrah*

Choose a policy below, then run the notebook from top to bottom. It

1. trains the policy with the settings of `configs/pusht.py` (or loads a saved checkpoint),
2. evaluates it on held-out episodes with the protocol of the paper,
3. shows a GIF of nine episodes, their final configurations, and the action chunks that the policy samples for one observation.

Training times on one GPU: OTP about 10 min, FMP and KRP about 10-20 min, SIP and DPR about 30-60 min, DP and CP several hours. The same functions are available from the terminal (`scripts/train.py`, `scripts/evaluate.py`).

In [ ]:
import os, sys, json
os.environ.setdefault('SDL_VIDEODRIVER', 'dummy')            # headless rendering of Push-T
ROOT = os.path.abspath('..')                                  # the repository root
sys.path.insert(0, ROOT)

import numpy as np
import matplotlib.pyplot as plt
import torch
from IPython.display import Image as ShowImage, display

import experiment as X
import visualization as V
from policies import NAMES
fontsize = 11
labeling = True                                               # set False to hide axis labels

## Settings

In [ ]:
POLICY     = 'otp'      # otp | fmp | sip | krp | dp | dpr | cp
CP_STEPS   = 1          # CP only: 1 (CP 1-step) or 3 (CP 3-step)
MODE       = 'train'    # 'train': train and save a checkpoint; 'load': use checkpoints/pusht/<policy>_seed<k>.pt
SEED       = 0          # training seed
N_EPISODES = 50         # held-out episodes 10000, 10001, ...; the paper uses 500 per seed
DEVICE     = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

cfg = X.get_config('pusht', POLICY)
print(NAMES[POLICY], 'settings:', cfg)

## Train (or load)

In [ ]:
losses = []
def log(line):
    """Print the training log and keep the losses for the plot below."""
    print(line)
    if 'iteration' in line:
        losses.append((int(line.split()[1].split('/')[0]), float(line.split()[-1])))

ckpt = X.checkpoint_path('pusht', POLICY, SEED)
if MODE == 'train':
    ckpt, seconds = X.train('pusht', POLICY, seed=SEED, device=DEVICE, log=log)
assert os.path.exists(ckpt), 'no checkpoint at %s: set MODE = "train"' % ckpt
print('checkpoint:', ckpt)

In [ ]:
if losses:
    it, val = np.array(losses).T
    plt.figure(figsize=(4, 3))
    plt.plot(it, val, 'o-', color='C0', lw=2)
    if labeling: plt.xlabel('iteration', fontsize=fontsize)
    if labeling: plt.ylabel('training loss (last term logged)', fontsize=fontsize)
    plt.tight_layout(); plt.show()

## Evaluate
The score of an episode is the maximum coverage ratio of the target (1 = task completed). Success: score 1; failure: score below 0.5.

In [ ]:
agent, cfg = X.make_agent('pusht', POLICY, ckpt, DEVICE, cp_steps=CP_STEPS)
episodes = X.task_module('pusht').HELD_OUT_SEEDS[:N_EPISODES]
res = X.run_episodes('pusht', agent, cfg, episodes,
                     progress=lambda k, n, s: print('episode %d/%d  score %.3f' % (k, n, s)) if k % 10 == 0 else None)
summary = X.summarize(res['score'])
print('%s: score %.3f +- %.3f, success %.1f%%, failure %.1f%%, %.2f s/episode (%d episodes)'
      % (NAMES[POLICY], summary['mean'], summary['se'], 100 * summary['success_rate'],
         100 * summary['failure_rate'], np.mean(res['seconds']), summary['n']))

name = X.tag(POLICY, SEED, cp_steps=CP_STEPS)
out_dir = os.path.join(ROOT, 'results', 'pusht', 'notebook')        # kept apart from scripts/evaluate.py's files
os.makedirs(out_dir, exist_ok=True)
res.pop('frames'); res.pop('rewards')
json.dump(dict(res, summary=summary), open(os.path.join(out_dir, name + '.json'), 'w'))

In [ ]:
plt.figure(figsize=(4, 3))
plt.hist(res['score'], bins=np.linspace(0, 1, 21), color='C0')
if labeling: plt.xlabel('episode score', fontsize=fontsize)
if labeling: plt.ylabel('episodes', fontsize=fontsize)
plt.tight_layout(); plt.show()

## Visualize
Nine held-out episodes, rendered: the T block (gray), its target (green), and the end effector (blue). The number in each episode is its score so far; at the end, a green frame marks a completed task and a red frame a failure.

In [ ]:
show = X.task_module('pusht').HELD_OUT_SEEDS[:9]
vis = X.run_episodes('pusht', agent, cfg, show, render=True)
gif = os.path.join(out_dir, name + '.gif')
V.grid_gif(vis['frames'], vis['rewards'], gif, title='%s, %d held-out episodes' % (NAMES[POLICY], len(show)))
display(ShowImage(filename=gif))

In [ ]:
# final configurations of the same episodes
display(V.final_frames(vis['frames'], vis['score']))

Action chunks sampled by the policy for the first observation of an episode (64 samples; red: the first action of each chunk). The spread of the chunks shows the conditional distribution that the policy represents.

In [ ]:
env = X.task_module('pusht').make_env()
env.seed(show[0])
obs = np.asarray(env.reset())[:20]
agent.reset(show[0])
plt.figure(figsize=(4, 4))
V.pusht_chunks(agent, np.stack([obs] * cfg['h_o']), env.render(mode='rgb_array'))
plt.title('%s: 64 sampled action chunks' % NAMES[POLICY], fontsize=fontsize)
plt.show()